# Classification with PyTorch: Baseball Hall of Fame
Predict if an MLB position player made the Hall of Fame (1) or not (0) from career batting stats.
Data: Lahman Baseball Database (Batting, Appearances, People, HallOfFame tables).

In [ ]:
!pip install -q mlxtend onnx onnxruntime xgboost

In [ ]:
import numpy as np
import torch
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from mlxtend.plotting import heatmap
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score
import xgboost as xgb
import onnxruntime as rt
import os
os.makedirs("docs", exist_ok=True)

In [ ]:
batch_size    = 32
learning_rate = 0.001
N_Epochs      = 100
epsilon       = 0.0001
torch.manual_seed(42)

## Read the data

In [ ]:
url = "https://raw.githubusercontent.com/cbwinslow/baseballdatabank/master/"
batting     = pd.read_csv(url + "core/Batting.csv")
appearances = pd.read_csv(url + "core/Appearances.csv")
people      = pd.read_csv(url + "core/People.csv")
hall        = pd.read_csv(url + "contrib/HallOfFame.csv")
print(batting.shape, appearances.shape, people.shape, hall.shape)

In [ ]:
batting.head()

## Process the data
1. Add up each player's seasons into career totals.
2. Keep position players only (remove pitchers).
3. Keep players with 10+ seasons (needed to be on the ballot) who retired by 2003, so their voting is finished.
4. Label = 1 if inducted as a player.

In [ ]:
## career totals
career = batting.groupby("playerID").agg(
    seasons=("yearID", "nunique"), last_season=("yearID", "max"),
    G=("G", "sum"), AB=("AB", "sum"), R=("R", "sum"), H=("H", "sum"),
    HR=("HR", "sum"), RBI=("RBI", "sum"), SB=("SB", "sum"), BB=("BB", "sum"))
career["AVG"] = career["H"] / career["AB"]

In [ ]:
## remove pitchers (players who pitched in at least half their games)
games = appearances.groupby("playerID")[["G_all", "G_p"]].sum()
career = career.join(games)
career = career[career["G_p"] < 0.5 * career["G_all"]]

In [ ]:
## eligible players: 10+ seasons, retired by 2003
career = career[(career["seasons"] >= 10) & (career["last_season"] <= 2003)]

## remove players banned from the Hall of Fame
career = career.drop(["rosepe01", "jacksjo01"], errors="ignore")

In [ ]:
## label: inducted as a player
inducted = set(hall[(hall["inducted"] == "Y") & (hall["category"] == "Player")]["playerID"])
## the HallOfFame table ends in 2018, so add the position players elected 2019-2025 who are in our date range
inducted |= {"baineha01", "simmote01", "hodgegi01", "minosmi01", "olivato01", "allendi01", "parkeda01"}
career["HOF"] = career.index.isin(inducted).astype(int)

In [ ]:
people["name"] = people["nameFirst"] + " " + people["nameLast"]
career = career.join(people.set_index("playerID")["name"])

headers_list = ["G", "R", "H", "HR", "RBI", "SB", "BB", "AVG", "HOF"]
HOF_data = career[headers_list + ["name"]].dropna()
print(HOF_data.shape)
HOF_data["HOF"].value_counts()

In [ ]:
HOF_data.sort_values("H", ascending=False).head(10)

In [ ]:
cm = np.corrcoef(HOF_data[headers_list].values.T)
hm = heatmap(cm, row_names=headers_list, column_names=headers_list, figsize=(10, 8))
plt.show()

In [ ]:
## Convert Pandas to Numpy
X = HOF_data[headers_list[:-1]].to_numpy()
y = HOF_data["HOF"].to_numpy()
names = HOF_data["name"].to_numpy()
print(X.shape)
print(y.shape)

In [ ]:
X_train, X_test, y_train, y_test, names_train, names_test = train_test_split(
    X, y, names, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
## fix data type
X_train = X_train.astype(np.float32)
X_test  = X_test.astype(np.float32)
y_train = y_train.astype(np.int64)
y_test  = y_test.astype(np.int64)

X_train_tr = torch.from_numpy(X_train)
X_test_tr  = torch.from_numpy(X_test)
y_train_tr = torch.from_numpy(y_train)
y_test_tr  = torch.from_numpy(y_test)

In [ ]:
x_means      = X_train_tr.mean(0, keepdim=True)
x_deviations = X_train_tr.std(0, keepdim=True) + epsilon

In [ ]:
train_ds = TensorDataset(X_train_tr, y_train_tr)
train_dl = DataLoader(train_ds, batch_size, shuffle=True)

## Models (2 outputs, one per class)

In [ ]:
## Logistic Regression
class LogReg_Net(nn.Module):
    def __init__(self, x_means, x_deviations):
        super().__init__()
        self.x_means      = x_means
        self.x_deviations = x_deviations
        self.linear1 = nn.Linear(8, 2)

    def forward(self, x):
        x = (x - self.x_means) / self.x_deviations
        y_pred = self.linear1(x)
        return y_pred

In [ ]:
## MLP
class MLP_Net(nn.Module):
    def __init__(self, x_means, x_deviations):
        super().__init__()
        self.x_means      = x_means
        self.x_deviations = x_deviations
        self.linear1 = nn.Linear(8, 16)
        self.act1    = nn.Sigmoid()
        self.linear2 = nn.Linear(16, 2)

    def forward(self, x):
        x = (x - self.x_means) / self.x_deviations
        x = self.linear1(x)
        x = self.act1(x)
        y_pred = self.linear2(x)
        return y_pred

In [ ]:
## Deep Learning with 2 hidden layers
class DL_Net(nn.Module):
    def __init__(self, x_means, x_deviations):
        super().__init__()
        self.x_means      = x_means
        self.x_deviations = x_deviations
        self.linear1 = nn.Linear(8, 32)
        self.act1    = nn.ReLU()
        self.linear2 = nn.Linear(32, 16)
        self.act2    = nn.ReLU()
        self.linear3 = nn.Linear(16, 2)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = (x - self.x_means) / self.x_deviations
        x = self.linear1(x)
        x = self.act1(x)
        x = self.dropout(x)
        x = self.linear2(x)
        x = self.act2(x)
        y_pred = self.linear3(x)
        return y_pred

## Training

In [ ]:
def training_loop(N_Epochs, model, loss_fn, opt):
    for epoch in range(N_Epochs):
        model.train()
        for xb, yb in train_dl:
            y_pred = model(xb)
            loss   = loss_fn(y_pred, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
        if epoch % 20 == 0:
            print(epoch, "loss=", loss.item())

In [ ]:
def print_metrics(y_true, y_pred):
    print("Accuracy:  %.3f" % accuracy_score(y_true, y_pred))
    print("Precision: %.3f" % precision_score(y_true, y_pred))
    print("Recall:    %.3f" % recall_score(y_true, y_pred))
    print("F1-score:  %.3f" % f1_score(y_true, y_pred))
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

In [ ]:
models = {
    "LogReg":        LogReg_Net(x_means, x_deviations),
    "MLP":           MLP_Net(x_means, x_deviations),
    "DL (2 hidden)": DL_Net(x_means, x_deviations),
}

results = []
for name, model in models.items():
    print("==== Training", name)
    opt = torch.optim.Adam(model.parameters(), lr=learning_rate)
    training_loop(N_Epochs, model, nn.CrossEntropyLoss(), opt)
    model.eval()
    y_pred_test = torch.argmax(model(X_test_tr), dim=1).numpy()
    print_metrics(y_test, y_pred_test)
    results.append([name, accuracy_score(y_test, y_pred_test), precision_score(y_test, y_pred_test),
                    recall_score(y_test, y_pred_test), f1_score(y_test, y_pred_test)])

## XGBoost (for comparison)

In [ ]:
clf = xgb.XGBClassifier(n_estimators=100, max_depth=3)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print_metrics(y_test, y_pred)
results.append(["XGBoost", accuracy_score(y_test, y_pred), precision_score(y_test, y_pred),
                recall_score(y_test, y_pred), f1_score(y_test, y_pred)])

## Results

In [ ]:
results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "Precision", "Recall", "F1"]).sort_values("F1", ascending=False)
results_df

In [ ]:
## best PyTorch model by F1
best_name  = max(models, key=lambda n: results_df.set_index("Model").loc[n, "F1"])
best_model = models[best_name]
print("Best PyTorch model:", best_name)

y_pred_test = torch.argmax(best_model(X_test_tr), dim=1).numpy()
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_test), display_labels=["not HOF", "HOF"]).plot(cmap="Blues")
plt.title(best_name)
plt.show()

In [ ]:
## which players did the model get wrong?
probs = F.softmax(best_model(X_test_tr), dim=1)[:, 1].detach().numpy()
wrong = pd.DataFrame({"name": names_test, "real": y_test, "pred": y_pred_test, "P(HOF)": probs.round(3)})
wrong[wrong["real"] != wrong["pred"]].sort_values("P(HOF)", ascending=False)

## Export to ONNX
Softmax is added at the end so the web page gets probabilities: [P(not HOF), P(HOF)].

In [ ]:
class WithSoftmax(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return F.softmax(self.model(x), dim=1)

export_model = WithSoftmax(best_model).eval().float()
dummy_input = torch.randn(1, 8, dtype=torch.float32)

torch.onnx.export(
    export_model,
    dummy_input,
    "docs/hof_classifier.onnx",
    input_names=["input1"],
    output_names=["output1"],
    opset_version=15,
    do_constant_folding=True,
    dynamo=False,
    dynamic_axes={"input1": {0: "batch"}, "output1": {0: "batch"}}
)
print("ONNX model saved")

In [ ]:
## check the ONNX model gives the same F1
sess = rt.InferenceSession("docs/hof_classifier.onnx")
p = sess.run(["output1"], {"input1": X_test})[0]
print("ONNX F1:", f1_score(y_test, p.argmax(1)))